In [1]:
import warnings
warnings.filterwarnings('ignore')
from datetime import datetime
import pandas as pd
import time
import requests
import json
import os
from dotenv import load_dotenv
import io
import pytz
from matplotlib import pyplot as plt

In [2]:
load_dotenv()
api_key = os.getenv("API_KEY_V1")

In [3]:
def get_reach_list_and_geometry_for_huc(huc):
    with open(f'data/huc_{huc}_geometries.geojson') as f:
        data = json.load(f)
        reach_id_list = [feature['id'] for feature in data['features']]
    return reach_id_list

huc_0305010505_reach_id_list = get_reach_list_and_geometry_for_huc("0305010505")
huc_030501050505_reach_id_list = get_reach_list_and_geometry_for_huc("030501050505")
print(f"Number of reaches in HUC 0305010505: {len(huc_0305010505_reach_id_list)}")
print(f"Number of reaches in HUC 030501050505: {len(huc_030501050505_reach_id_list)}")

Number of reaches in HUC 0305010505: 115
Number of reaches in HUC 030501050505: 10


In [4]:
reach_mapper_dict = {
    "HUC-0305010505": ','.join(map(str, huc_0305010505_reach_id_list)),
    "HUC-030501050505": ','.join(map(str, huc_030501050505_reach_id_list)),
    "Reach-12034579": '12034579',
}

In [5]:
reach_mapper_dict["HUC-0305010505"]

'12035527,12035387,12034663,12034723,12034759,12034711,12035545,12034443,12034627,12034481,12035405,12034877,12035547,12034679,12035389,12035815,12034397,12034701,12035541,12034719,166756451,12034735,12034673,12034453,12034795,12034527,12034793,12035513,12035805,12034779,12035777,12035523,12035515,12034797,12034403,12034451,12034751,12035947,12034697,12034473,12035803,12034801,12035833,12035517,12035565,12034577,12035531,12034541,12034507,12035553,12035407,12034893,12035779,12035379,12035393,25776238,12034699,12035557,12034579,12034727,12034503,12034773,12034665,12034635,12034513,12034819,12034767,12035509,12034707,12035381,12034741,12035537,12034815,166756450,12035525,12034887,12035561,12034539,12034501,12035543,12035529,12035519,12034743,12034341,12034563,12035533,12034471,12035395,12034713,12034709,12034683,12034731,12034547,12034545,12034485,12034499,12034775,12035783,12035949,12035539,12034813,12035511,12034561,12035809,12035559,12034737,12034401,12035521,12035555,12035535,1203445

In [6]:
API_ROOT_URL = 'https://nwm-api.ciroh.org'
FORECASTS = f'{API_ROOT_URL}/forecast'
ANALYSES_ASSIM = f'{API_ROOT_URL}/analysis-assim'
RETURN_PERIODS = f'{API_ROOT_URL}/return-period'
GEOMETRIES = f'{API_ROOT_URL}/geometry'

In [7]:
def get_nwm_data(endpoint, params):
    start = time.perf_counter()
    r = requests.get(url=endpoint, params=params, 
                     headers={
                         'x-api-key': api_key
                         })
    if r.status_code == 200:
        if 'output_format' in params.keys():
            if params['output_format'] == 'csv':
                df = pd.read_csv(io.StringIO(r.text))
                print("Data dimension:", df.shape)
                end = time.perf_counter()
                elaspse = end - start
                mst_timezone = pytz.timezone('America/Denver')
                utc_now = datetime.now(pytz.utc)
                mst_now = utc_now.astimezone(mst_timezone)
                print("Job timestamp (MST):", mst_now.strftime("%Y-%m-%d %H:%M:%S"))
                return elaspse
            elif params['output_format'] == 'json':
                json_data = json.loads(r.text)
                print("Data dimension:", len(json_data))
                end = time.perf_counter()
                elaspse = end - start
                mst_timezone = pytz.timezone('America/Denver')
                utc_now = datetime.now(pytz.utc)
                mst_now = utc_now.astimezone(mst_timezone)
                print("Job timestamp (MST):", mst_now.strftime("%Y-%m-%d %H:%M:%S"))
                return elaspse
    else:
        print("HTTP Error:", r.text)

In [8]:
def analysis_case_run(case, end_date):
    with open(f'results/nwmapiv1/analysis_{case}.json', "w") as f:
        for label, reach_id_list_str in reach_mapper_dict.items():
            elapsed = get_nwm_data(
                endpoint = ANALYSES_ASSIM,
                params = {
                    'start_time': "2024-01-01T00:00:00",
                    'end_time': end_date,
                    'comids': reach_id_list_str,
                    'output_format': 'csv'
                })
            elapsed_time_dict = {
                "label": label,
                "elapsed_time_sec": elapsed
            }
            print(elapsed_time_dict)
            print("_" * 40)
            f.write(json.dumps(elapsed_time_dict) + "\n")

In [9]:
# Analysis Case 1: 1 week
analysis_case_run("case_01", "2024-01-08T00:00:00")

HTTP Error: {"code":504,"message":"upstream request timeout"}

{'label': 'HUC-0305010505', 'elapsed_time_sec': None}
________________________________________
Data dimension: (1690, 4)
Job timestamp (MST): 2026-04-07 20:46:59
{'label': 'HUC-030501050505', 'elapsed_time_sec': 9.444745291984873}
________________________________________
Data dimension: (169, 4)
Job timestamp (MST): 2026-04-07 20:47:06
{'label': 'Reach-12034579', 'elapsed_time_sec': 6.457419334008591}
________________________________________


In [10]:
# Analysis Case 2: 1 month
analysis_case_run("case_02", "2024-02-01T00:00:00")

HTTP Error: {"code":504,"message":"upstream request timeout"}

{'label': 'HUC-0305010505', 'elapsed_time_sec': None}
________________________________________
Data dimension: (7450, 4)
Job timestamp (MST): 2026-04-07 20:47:44
{'label': 'HUC-030501050505', 'elapsed_time_sec': 10.528941333002876}
________________________________________
Data dimension: (745, 4)
Job timestamp (MST): 2026-04-07 20:47:55
{'label': 'Reach-12034579', 'elapsed_time_sec': 10.297172624996165}
________________________________________


In [11]:
def srf_case_run(case):
    with open(f'results/nwmapiv1/short_range_{case}.json', "w") as f:
        for label, reach_id_list_str in reach_mapper_dict.items():
            elapsed = get_nwm_data(
                endpoint = FORECASTS,
                params = {
                    'forecast_type': 'short_range',
                    'reference_time': "2024-01-01T00:00:00",
                    'comids': reach_id_list_str,
                    'output_format': 'csv'
                })
            elapsed_time_dict = {
                "label": label,
                "elapsed_time_sec": elapsed
            }
            print(elapsed_time_dict)
            print("_" * 40)
            f.write(json.dumps(elapsed_time_dict) + "\n")          

In [12]:
# Short-range Case 1
srf_case_run("case_01")

Data dimension: (2070, 6)
Job timestamp (MST): 2026-04-07 20:48:16
{'label': 'HUC-0305010505', 'elapsed_time_sec': 4.859883834025823}
________________________________________
Data dimension: (180, 6)
Job timestamp (MST): 2026-04-07 20:48:19
{'label': 'HUC-030501050505', 'elapsed_time_sec': 2.3744804580055643}
________________________________________
Data dimension: (18, 6)
Job timestamp (MST): 2026-04-07 20:48:21
{'label': 'Reach-12034579', 'elapsed_time_sec': 2.321486000000732}
________________________________________


In [13]:
def mrf_case_run(case, ensemble):
    with open(f'results/nwmapiv1/medium_range_{case}.json', "w") as f:
        for label, reach_id_list_str in reach_mapper_dict.items():
            elapsed = get_nwm_data(
                endpoint = FORECASTS,
                params = {
                    'forecast_type': 'medium_range',
                    'reference_time': "2024-01-01T00:00:00",
                    'comids': reach_id_list_str,
                    'ensemble': ensemble,
                    'output_format': 'csv'
                })
            elapsed_time_dict = {
                "label": label,
                "elapsed_time_sec": elapsed
            }
            print(elapsed_time_dict)
            print("_" * 40)
            f.write(json.dumps(elapsed_time_dict) + "\n")

In [14]:
# Medium-range Case 1: Single ensemble member (member 3)
mrf_case_run("case_01", ensemble="3")

HTTP Error: {"code":504,"message":"upstream request timeout"}

{'label': 'HUC-0305010505', 'elapsed_time_sec': None}
________________________________________
Data dimension: (2040, 6)
Job timestamp (MST): 2026-04-07 20:49:05
{'label': 'HUC-030501050505', 'elapsed_time_sec': 11.971988334000343}
________________________________________
Data dimension: (204, 6)
Job timestamp (MST): 2026-04-07 20:49:18
{'label': 'Reach-12034579', 'elapsed_time_sec': 12.699501166993286}
________________________________________


In [15]:
# Medium-range Case 2: Average of all ensemble members
mrf_case_run("case_02", ensemble=None)

Data dimension: (27600, 6)
Job timestamp (MST): 2026-04-07 20:49:36
{'label': 'HUC-0305010505', 'elapsed_time_sec': 14.835097208997468}
________________________________________
Data dimension: (2400, 6)
Job timestamp (MST): 2026-04-07 20:49:47
{'label': 'HUC-030501050505', 'elapsed_time_sec': 10.72043295900221}
________________________________________
Data dimension: (240, 6)
Job timestamp (MST): 2026-04-07 20:49:57
{'label': 'Reach-12034579', 'elapsed_time_sec': 9.439253208984155}
________________________________________


In [16]:
def lrf_case_run(case, ensemble):
    with open(f'results/nwmapiv1/long_range_{case}.json', "w") as f:
        for label, reach_id_list_str in reach_mapper_dict.items():
            elapsed = get_nwm_data(
                endpoint = FORECASTS,
                params = {
                    'forecast_type': 'long_range',
                    'reference_time': "2024-01-01T00:00:00",
                    'comids': reach_id_list_str,
                    'ensemble': ensemble,
                    'output_format': 'csv'
                })
            elapsed_time_dict = {
                "label": label,
                "elapsed_time_sec": elapsed
            }
            print(elapsed_time_dict)
            print("_" * 40)
            f.write(json.dumps(elapsed_time_dict) + "\n")

In [17]:
# LRF Case 1: Single ensemble member (member 0)
lrf_case_run("case_01", ensemble="0")

Data dimension: (13800, 6)
Job timestamp (MST): 2026-04-07 20:50:20
{'label': 'HUC-0305010505', 'elapsed_time_sec': 14.056048209022265}
________________________________________
Data dimension: (1200, 6)
Job timestamp (MST): 2026-04-07 20:50:25
{'label': 'HUC-030501050505', 'elapsed_time_sec': 5.2447953750088345}
________________________________________
Data dimension: (120, 6)
Job timestamp (MST): 2026-04-07 20:50:30
{'label': 'Reach-12034579', 'elapsed_time_sec': 4.526825624983758}
________________________________________


In [18]:
# LRF Case 2: Average of all ensemble members
lrf_case_run("case_02", ensemble=None)

Data dimension: (13800, 6)
Job timestamp (MST): 2026-04-07 20:51:01
{'label': 'HUC-0305010505', 'elapsed_time_sec': 7.987775499990676}
________________________________________
Data dimension: (1200, 6)
Job timestamp (MST): 2026-04-07 20:51:06
{'label': 'HUC-030501050505', 'elapsed_time_sec': 5.415880250016926}
________________________________________
Data dimension: (120, 6)
Job timestamp (MST): 2026-04-07 20:51:11
{'label': 'Reach-12034579', 'elapsed_time_sec': 4.838086083997041}
________________________________________


In [19]:
def return_period_case_run(case, return_periods=None):
    with open(f'results/nwmapiv1/return_period_{case}.json', "w") as f:
        for label, reach_id_list_str in reach_mapper_dict.items():
            elapsed = get_nwm_data(
                endpoint = RETURN_PERIODS,
                params = {
                    'comids': reach_id_list_str,
                    'output_format': 'csv',
                    'return_periods': return_periods
                })
            elapsed_time_dict = {
                "label": label,
                "elapsed_time_sec": elapsed
            }
            print(elapsed_time_dict)
            print("_" * 40)
            f.write(json.dumps(elapsed_time_dict) + "\n")

In [20]:
# return period case: all return periods
return_period_case_run("case_01")

Data dimension: (115, 7)
Job timestamp (MST): 2026-04-07 20:51:19
{'label': 'HUC-0305010505', 'elapsed_time_sec': 1.4272745419875719}
________________________________________
Data dimension: (10, 7)
Job timestamp (MST): 2026-04-07 20:51:21
{'label': 'HUC-030501050505', 'elapsed_time_sec': 1.5484673329920042}
________________________________________
Data dimension: (1, 7)
Job timestamp (MST): 2026-04-07 20:51:22
{'label': 'Reach-12034579', 'elapsed_time_sec': 1.0192125839821529}
________________________________________


In [21]:
def geometry_case_run(case):
    with open(f'results/nwmapiv1/geometry_{case}.json', "w") as f:
        for label, reach_id_list_str in reach_mapper_dict.items():
            elapsed = get_nwm_data(
                endpoint = GEOMETRIES,
                params = {
                    'comids': reach_id_list_str,
                    'output_format': 'csv'
                })
            elapsed_time_dict = {
                "label": label,
                "elapsed_time_sec": elapsed
            }
            print(elapsed_time_dict)
            print("_" * 40)
            f.write(json.dumps(elapsed_time_dict) + "\n")

In [22]:
# Case 1
geometry_case_run("case_01")

Data dimension: (115, 5)
Job timestamp (MST): 2026-04-07 20:51:32
{'label': 'HUC-0305010505', 'elapsed_time_sec': 2.4938270410057157}
________________________________________
Data dimension: (10, 5)
Job timestamp (MST): 2026-04-07 20:51:33
{'label': 'HUC-030501050505', 'elapsed_time_sec': 1.358502832998056}
________________________________________
Data dimension: (1, 5)
Job timestamp (MST): 2026-04-07 20:51:35
{'label': 'Reach-12034579', 'elapsed_time_sec': 1.5300733750045765}
________________________________________
